In [ ]:
import pandas as pd

In [ ]:
df=pd.read_csv("IMDB Dataset.csv")

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df["review"].isnull().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.shape

## pre-processing

### 1.converting to lowercase

In [ ]:

df["review"]=df["review"].str.lower()

### 2.removing the URLs

In [ ]:
import re

# sample_text="abc is the word, abc" #abc=>xyz

# new_text=re.sub("abc","xyz",sample_text)

In [ ]:
def remove_urls(text):
    text=re.sub(r"http\S+", "",text) #(pattern,repl,string)     \S=> no white space characters, \s=>white space
    return text
df["review"]=df["review"].apply(remove_urls)

### 3.remove punctuations

In [ ]:
def remove_punctuations(text):
    text=re.sub(r"[^A-Za-z0-9\s]","",text) #A-Z,a-z,0-9,space(\s)=>values outside this is considered punctuations
    return text

df["review"]=df["review"].apply(remove_punctuations)

In [ ]:
### 4.removing html tags

def remove_htmltags(text):
    text=re.sub(r"<.*?>","",text) 
    return text

df["review"]=df["review"].apply(remove_htmltags)

### 5.removing the stopwords

In [ ]:
%pip install nltk
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [ ]:
#word_tokenize(understanding with an example)
# sample_text="i like coding in python!:)"
# tokens=word_tokenize(sample_text)

In [ ]:
def remove_stopwords(text):
    tokens=word_tokenize(text)
    stop_words=stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text.replace(word,"")

    return text

df["review"]=df["review"].apply(remove_stopwords)

In [ ]:
df.head()

### 6.stemming

In [ ]:
#running=>run
#playes=>play

from nltk.stem import PorterStemmer

def stremming(text):
    ps=PorterStemmer()
    stemmed_words=[]

    tokens=word_tokenize(text)
    for token in tokens:
        stemmed_token=ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"]=df["review"].apply(stremming)

In [ ]:
df.head()

### 7.Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

le=LabelEncoder()

df["sentiment"]=le.fit_transform(df["sentiment"])

In [ ]:
y=df["sentiment"]

In [ ]:
y

### 8.vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf=TfidfVectorizer(max_features=5000)

X=tf.fit_transform(df["review"])

In [ ]:
X

## dataset and dataloaders

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [ ]:
X_test

In [ ]:
import torch
from torch.utils.data import TensorDataset,DataLoader

In [ ]:
X_train=X_train.toarray()
X_test=X_test.toarray()

In [ ]:
X_train

In [ ]:
train_set=TensorDataset(
    torch.from_numpy(X_train).float(), #creating tensors
    torch.from_numpy(y_train.values).float()
)

test_set=TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [ ]:
train_loader=DataLoader(train_set,shuffle=True,batch_size=64)
test_loader=DataLoader(test_set,shuffle=True,batch_size=64)

## build our RNN

In [ ]:
import torch.nn as nn
import torch.optim as optim

In [ ]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layers=1):
        super().__init__()

        self.hidden_size=hidden_size
        self.num_layers=num_layers


        #rnnlayer
        self.rnn=nn.RNN(input_size,hidden_size,num_layers,batch_first=True)

        #fully connected layer
        self.fc=nn.Linear(hidden_size,1) #we are building many-to-one architecture

    def forward(self,x):
        #optional
        h0=torch.zeros(self.num_layers,x.size(0),self.hidden_size)

        out,_=self.rnn(x,h0)

        #1st value=>hidden state of all timesteps
        #2nd vlaue=>final hidden state of last timestep

        outputs=self.fc(out[:,-1,:])
        return outputs

In [ ]:
input_size=X_train.shape[1]

model=RNN(input_size)

criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters())

## training the rnn

In [ ]:
# unsqueeze(brings an additional dimention into data),squeeze(removes a dimention)

epochs=10

for epoch in range(epochs):

    model.train()

    for xb,yb in train_loader:
        optimizer.zero_grad()

        xb=xb.unsqueeze(1) #add singleton direction
        
        outputs=model(xb)#(batch_size,1)

        outputs=torch.sigmoid(outputs.squeeze())#(batch_size,) => probablity

        loss=criterion(outputs,yb) #compute loss
        loss.backward()
        optimizer.step() #weight update

    print(f"epoch={epoch+1}/{epochs}and loss={loss.item()}")

In [ ]:
# evaluate

model.eval()

with torch.no_grad():
    correct_values=0
    total_values=0
    
    for xb,yb in test_loader:
        xb=xb.unsqueeze(1)

        outputs=model(xb)
        predicted=(torch.sigmoid(outputs.squeeze()) > 0.5).float()
        total_values+=yb.size(0)
        correct_values+=(predicted==yb).sum().item()

    print(f"accuracy={correct_values/total_values*100}")